In [5]:
from pathlib import Path
import pandas as pd



current_dir = Path.cwd()


repo_path = current_dir
print(repo_path)

while repo_path != repo_path.parent:
    if (
        (repo_path / "train_split.csv").exists()
        and (repo_path / "test_split.csv").exists()
    ):
        break

    repo_path = repo_path.parent


train_path = repo_path / "train_split.csv"
test_path = repo_path / "test_split.csv"
print(train_path)


if not train_path.exists():
    raise FileNotFoundError(
        f"train_split.csv was not found.\n"
        f"Expected location: {train_path}"
    )

if not test_path.exists():
    raise FileNotFoundError(
        f"test_split.csv was not found.\n"
        f"Expected location: {test_path}"
    )

print(f"Repository path: {repo_path.resolve()}")
print(f"Training dataset: {train_path.resolve()}")
print(f"Testing dataset:  {test_path.resolve()}")

/content
/content/train_split.csv
Repository path: /content
Training dataset: /content/train_split.csv
Testing dataset:  /content/test_split.csv


In [6]:
# ---------------------------------------------------------
# Load datasets
# ---------------------------------------------------------

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("\n--- Data Loaded Successfully ---")

print(f"Training shape: {train_df.shape}")
print(f"Testing shape:  {test_df.shape}")

print("\nTraining data:")
display(train_df.head())

print("\nTesting data:")
display(test_df.head())


--- Data Loaded Successfully ---
Training shape: (6400, 2)
Testing shape:  (1600, 2)

Training data:


,clean_text,label
0,nearly year thailand devalued baht setting wor...,Not Relevant
1,new york investors try play stock market hitch...,Not Relevant
2,richmond march fierce winter weather impact na...,Relevant
3,washington debtor nations bankers scramble def...,Not Relevant
4,loud voices screaming circles deficit sleeper ...,Not Relevant



Testing data:


,clean_text,label
0,little cither jiational state level determine ...,Not Relevant
1,washington roughly americans work looking jobs...,Relevant
2,weekday erie neighborhood house citys north fe...,Not Relevant
3,candidate says win basis fundamental economic ...,Not Relevant
4,regard home place live investment substitute r...,Not Relevant


In [7]:

print("Training columns:")
print(train_df.columns.tolist())

print("\nTesting columns:")
print(test_df.columns.tolist())

required_columns = {"clean_text", "label"}
missing_train = required_columns - set(train_df.columns)
missing_test = required_columns - set(test_df.columns)

if missing_train:
    raise KeyError(f"Missing columns in training data: {sorted(missing_train)}")

if missing_test:
    raise KeyError(f"Missing columns in testing data: {sorted(missing_test)}")

print("\nRequired columns found: clean_text and label")
print("\nLabel distribution in training data:")
print(train_df["label"].value_counts())


Training columns:
['clean_text', 'label']

Testing columns:
['clean_text', 'label']

Required columns found: clean_text and label

Label distribution in training data:
label
Not Relevant    5264
Relevant        1136
Name: count, dtype: int64



## Feature Extraction and Logistic Regression with Varying `max_features`

Bag-of-Words (BoW) features are extracted from `clean_text` using `CountVectorizer`.  
A Logistic Regression classifier is then trained for three vocabulary sizes: **40,000, 5,000, and 1,000 features**.

The test set is transformed using the vectorizer fitted on the training set, which prevents information from the test set from leaking into training.


In [8]:

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import numpy as np


X_train_text = train_df["clean_text"].fillna("")
X_test_text = test_df["clean_text"].fillna("")
y_train = train_df["label"]
y_test = test_df["label"]

max_features_list = [40000, 5000, 1000]
results = {}

for mf in max_features_list:
    print(f"\n{'=' * 70}")
    print(f"Processing max_features = {mf}")
    print(f"{'=' * 70}")


    vectorizer = CountVectorizer(max_features=mf)
    X_train_bow = vectorizer.fit_transform(X_train_text)


    X_test_bow = vectorizer.transform(X_test_text)

    print(f"Train BoW shape: {X_train_bow.shape}")
    print(f"Test BoW shape:  {X_test_bow.shape}")
    print(f"Actual vocabulary size: {len(vectorizer.vocabulary_)}")


    model = LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver="liblinear"
    )
    model.fit(X_train_bow, y_train)


    y_pred = model.predict(X_test_bow)


    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, average="weighted", zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, average="weighted", zero_division=0
    )

    results[mf] = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

    print("\n--- Model Evaluation ---")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    cm = confusion_matrix(y_test, y_pred)

    print("Confusion Matrix:")
    print(cm)
    print(f"True labels:      {np.unique(y_test).tolist()}")
    print(f"Model classes:    {model.classes_.tolist()}")

print("\n" + "=" * 70)
print("SUMMARY OF RESULTS")
print("=" * 70)

for mf, metrics in results.items():
    print(f"\nMax Features = {mf}")
    for metric, value in metrics.items():
        print(f"  {metric.replace('_', ' ').title()}: {value:.4f}")



Processing max_features = 40000
Train BoW shape: (6400, 40000)
Test BoW shape:  (1600, 40000)
Actual vocabulary size: 40000

--- Model Evaluation ---
Accuracy : 0.7850
Precision: 0.7629
Recall   : 0.7850
F1-Score : 0.7723

Classification Report:
              precision    recall  f1-score   support

Not Relevant       0.85      0.90      0.87      1316
    Relevant       0.36      0.27      0.31       284

    accuracy                           0.79      1600
   macro avg       0.60      0.58      0.59      1600
weighted avg       0.76      0.79      0.77      1600

Confusion Matrix:
[[1180  136]
 [ 208   76]]
True labels:      ['Not Relevant', 'Relevant']
Model classes:    ['Not Relevant', 'Relevant']

Processing max_features = 5000
Train BoW shape: (6400, 5000)
Test BoW shape:  (1600, 5000)
Actual vocabulary size: 5000

--- Model Evaluation ---
Accuracy : 0.7744
Precision: 0.7658
Recall   : 0.7744
F1-Score : 0.7699

Classification Report:
              precision    recall  f1-score 